# Wikipedia Data Collection Pipeline
Sahana Bai Sankarrao — MATH5872M

This notebook builds `wikipedia_final.csv` from the raw Wikipedia dump chunks: it extracts
the six structural features and real category tags directly from each article's wikitext,
fetches the real community-assigned quality label for each article from its Talk page via
the Wikipedia API, and merges these into the single labelled dataset used throughout
`Wikipedia_analysis_final.ipynb`.

Run this notebook first; its output (`wikipedia_final.csv`) is the input to the analysis
notebook. Fetching real quality labels from the API for ~1,400 articles takes roughly
15-20 minutes (Section 3 below) — the dump files themselves must be downloaded separately
from dumps.wikimedia.org and placed in this notebook's working directory.

## Section 1 — Setup

In [1]:
!pip install mwxml mwparserfromhell

In [2]:
import bz2
import re
import time
import requests
import pandas as pd
import mwxml
import mwparserfromhell

CHUNKS = [
    "enwiki-latest-pages-articles-multistream1.xml-p1p41242.bz2",
    "enwiki-latest-pages-articles-multistream2.xml-p41243p151573.bz2",
    "enwiki-latest-pages-articles-multistream3.xml-p151574p311329.bz2",
    "enwiki-latest-pages-articles-multistream4.xml-p311330p558391.bz2",
]
MAX_ARTICLES_PER_CHUNK = 500

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Wikipedia Quality Research; sahana.dissertation@leeds.ac.uk)"
}

## Section 2 — Extract structural features and topic categories from the dump chunks

For each of the four dump chunks, this steps through up to 500 pages, skips redirects,
and for every remaining article extracts:
- the six structural features (length, references, images, sections, internal links, infobox)
- the article's real Wikipedia category tags, mapped to one of six broad topic areas

This is the single pass over the raw wikitext; no quality label is assigned here — that
comes from the Talk page API in Section 3, kept deliberately independent of these
structural features to avoid circular reasoning.

In [3]:
def assign_category_from_cats(categories):
    """Map an article's real Wikipedia category tags to one of six broad topic areas,
    checked in a fixed order (Science & Technology first, General as the fallback)."""
    cats_text = " ".join(categories).lower()

    if any(w in cats_text for w in [
        "science", "physics", "chemistry", "biology", "mathematics",
        "astronomy", "geology", "ecology", "medicine", "technology",
        "computing", "software", "engineering", "electronics",
        "species", "animal", "plant", "bird", "insect", "fish",
        "mammal", "bacteria", "virus", "genetics", "evolution",
        "climate", "environment", "quantum", "atomic"
    ]):
        return "Science & Technology"

    elif any(w in cats_text for w in [
        "history", "war", "battle", "revolution", "empire", "dynasty",
        "ancient", "medieval", "century", "politics", "election",
        "government", "democracy", "republic", "parliament", "congress",
        "colonialism", "civilization", "military", "treaty", "independence",
        "political", "nationalism", "socialism", "communism"
    ]):
        return "History & Politics"

    elif any(w in cats_text for w in [
        "geography", "country", "city", "town", "river", "mountain",
        "island", "lake", "ocean", "continent", "region", "district",
        "province", "municipality", "populated places", "settlements",
        "states", "counties", "capitals", "administrative"
    ]):
        return "Geography"

    elif any(w in cats_text for w in [
        "birth", "death", "people", "born", "died", "biography",
        "politicians", "scientists", "artists", "musicians", "actors",
        "writers", "philosophers", "mathematicians", "engineers",
        "presidents", "kings", "queens", "monarchs", "generals"
    ]):
        return "Biography"

    elif any(w in cats_text for w in [
        "film", "music", "album", "song", "band", "art", "novel",
        "book", "literature", "poetry", "television", "sport",
        "football", "cricket", "olympic", "award", "culture",
        "religion", "philosophy", "mythology", "entertainment",
        "theatre", "dance", "painting", "sculpture", "architecture"
    ]):
        return "Culture & Arts"

    else:
        return "General"


def extract_features_and_topics(filename, max_articles=MAX_ARTICLES_PER_CHUNK):
    """Step through one dump chunk, skip redirects, and extract structural features
    plus topic category for every remaining article (one revision per page)."""
    articles = []
    pages_seen = 0

    print(f"Processing {filename}...")

    with bz2.open(filename, "rb") as f:
        dump = mwxml.Dump.from_file(f)

        for page in dump:
            for revision in page:
                text = revision.text or ""

                # Skip redirects -- they have no real content of their own
                if text.strip().startswith("#REDIRECT"):
                    break

                parsed = mwparserfromhell.parse(text)

                categories = [
                    str(link.title)
                    for link in parsed.filter_wikilinks()
                    if str(link.title).startswith("Category:")
                ]
                topic = assign_category_from_cats(categories)

                articles.append({
                    "title":      page.title,
                    "length":     len(text),
                    "refs":       text.count("<ref"),
                    "images":     text.count("[[File:") + text.count("[[Image:"),
                    "sections":   text.count("=="),
                    "links":      len(parsed.filter_wikilinks()),
                    "infobox":    1 if "{{Infobox" in text else 0,
                    "categories": "|".join(categories[:10]),
                    "topic":      topic,
                })
                break

            pages_seen += 1
            if pages_seen >= max_articles:
                break

    print(f"  Done -- {len(articles)} articles extracted ({pages_seen - len(articles)} redirects skipped)")
    return articles


all_articles = []
for chunk in CHUNKS:
    all_articles.extend(extract_features_and_topics(chunk))

df_topics = pd.DataFrame(all_articles).drop_duplicates(subset="title")

print(f"\nTotal articles extracted: {len(df_topics)}")
print("\nTopic distribution:")
print(df_topics["topic"].value_counts())

df_topics.to_csv("wikipedia_with_topics.csv", index=False)
print("\nSaved to wikipedia_with_topics.csv")

Processing enwiki-latest-pages-articles-multistream1.xml-p1p41242.bz2...
  Done -- 325 articles extracted (175 redirects skipped)
Processing enwiki-latest-pages-articles-multistream2.xml-p41243p151573.bz2...
  Done -- 386 articles extracted (114 redirects skipped)
Processing enwiki-latest-pages-articles-multistream3.xml-p151574p311329.bz2...
  Done -- 355 articles extracted (145 redirects skipped)
Processing enwiki-latest-pages-articles-multistream4.xml-p311330p558391.bz2...
  Done -- 339 articles extracted (161 redirects skipped)

Total articles extracted: 1405

Topic distribution:
topic
General                 593
History & Politics      267
Science & Technology    255
Geography               180
Culture & Arts           74
Biography                36
Name: count, dtype: int64

Saved to wikipedia_with_topics.csv


## Section 3 — Fetch real quality labels from Wikipedia Talk pages

For each article extracted above, query its Talk page via the Wikipedia API and search
the WikiProject banner / article history templates for a `class=` or `currentstatus=`
assignment. This label is obtained entirely independently of the six structural features
extracted in Section 2 -- it never looks at the article's own text, only its Talk page --
which avoids the circular-reasoning problem some prior work runs into (Moás & Lopes, 2023).

This takes roughly 15-20 minutes for ~1,400 articles at a polite 0.5s delay per request.

In [4]:
def get_real_quality_label(title):
    """Query the Talk page for `title` and extract its community-assigned quality
    class from WikiProject banner / article history templates. Returns "Unknown"
    if no recognisable label is found (e.g. disambiguation pages, list articles)."""
    try:
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "titles": "Talk:" + title,
            "prop": "revisions",
            "rvprop": "content",
            "rvslots": "main",
            "format": "json",
        }
        r = requests.get(url, params=params, headers=HEADERS, timeout=10)
        data = r.json()
        page = list(data["query"]["pages"].values())[0]

        if "revisions" not in page:
            return "Unknown"

        content = page["revisions"][0]["slots"]["main"]["*"].lower()

        if "currentstatus=fa" in content or "|class=fa" in content or "| class = fa" in content:
            return "FA"
        elif "currentstatus=ga" in content or "|class=ga" in content or "| class = ga" in content:
            return "GA"
        elif "|class=b" in content or "| class = b" in content:
            return "B"
        elif "|class=c" in content or "| class = c" in content:
            return "C"
        elif "|class=start" in content or "| class = start" in content:
            return "Start"
        elif "|class=stub" in content or "| class = stub" in content:
            return "Stub"
        else:
            return "Unknown"

    except Exception:
        return "Unknown"


df_topics = pd.read_csv("wikipedia_with_topics.csv")

real_labels = []
total = len(df_topics)
print(f"Fetching real labels for {total} articles...")

for i, title in enumerate(df_topics["title"]):
    real_labels.append(get_real_quality_label(title))
    if (i + 1) % 100 == 0:
        print(f"  Progress: {i + 1}/{total} articles done...")
    time.sleep(0.5)  # be polite to Wikipedia's servers

df_topics["quality_real"] = real_labels
df_topics.to_csv("wikipedia_features_labelled.csv", index=False)

print("\nDone! Saved to wikipedia_features_labelled.csv")
print("\nReal quality label distribution:")
print(df_topics["quality_real"].value_counts())
unknown_count = (df_topics["quality_real"] == "Unknown").sum()
print(f"\nUnknown count: {unknown_count}")

Fetching real labels for 1405 articles...
  Progress: 100/1405 articles done...
  Progress: 200/1405 articles done...
  Progress: 300/1405 articles done...
  Progress: 400/1405 articles done...
  Progress: 500/1405 articles done...
  Progress: 600/1405 articles done...
  Progress: 700/1405 articles done...
  Progress: 800/1405 articles done...
  Progress: 900/1405 articles done...
  Progress: 1000/1405 articles done...
  Progress: 1100/1405 articles done...
  Progress: 1200/1405 articles done...
  Progress: 1300/1405 articles done...
  Progress: 1400/1405 articles done...

Done! Saved to wikipedia_features_labelled.csv

Real quality label distribution:
quality_real
Start      394
C          338
Unknown    220
B          199
Stub       196
GA          43
FA          15
Name: count, dtype: int64

Unknown count: 220


## Section 4 — Clean the dataset and build the final labelled file

Articles with no recognisable quality label are mostly disambiguation pages, list articles
and navigation aids -- page types Wikipedia doesn't apply its quality assessment process to
in the first place -- so it's correct that they have no rating to extract. Drop them, add
the collapsed three-class label (Low / Medium / High), and save the final dataset used by
`Wikipedia_analysis_final.ipynb`.

In [5]:
df = pd.read_csv("wikipedia_features_labelled.csv")

unknown = df[df["quality_real"] == "Unknown"]
print(f"Dropping {len(unknown)} articles with no recognisable quality label")

df_final = df[df["quality_real"] != "Unknown"].copy().reset_index(drop=True)


def collapse_quality(q):
    if q in ["Stub", "Start"]:
        return "Low"
    elif q in ["C", "B"]:
        return "Medium"
    else:  # GA, FA
        return "High"


df_final["quality_3class"] = df_final["quality_real"].apply(collapse_quality)

print(f"\nFinal dataset: {len(df_final)} articles")
print("\nSix-class distribution:")
print(df_final["quality_real"].value_counts())
print("\nThree-class distribution:")
print(df_final["quality_3class"].value_counts())
print("\nTopic distribution:")
print(df_final["topic"].value_counts())

df_final.to_csv("wikipedia_final.csv", index=False)
print("\nSaved to wikipedia_final.csv -- ready for Wikipedia_analysis_final.ipynb")

Dropping 220 articles with no recognisable quality label

Final dataset: 1185 articles

Six-class distribution:
quality_real
Start    394
C        338
B        199
Stub     196
GA        43
FA        15
Name: count, dtype: int64

Three-class distribution:
quality_3class
Low       590
Medium    537
High       58
Name: count, dtype: int64

Topic distribution:
topic
General                 385
History & Politics      263
Science & Technology    252
Geography               179
Culture & Arts           71
Biography                35
Name: count, dtype: int64

Saved to wikipedia_final.csv -- ready for Wikipedia_analysis_final.ipynb
